# T1.1 – Road Collision Severity: Data Preparation & Feature Engineering

This notebook cleans the UK DfT road collision dataset, explores it visually, and engineers/tests candidate features (using Mutual Information) ahead of modelling.

## 1. Setup & Imports

In [ ]:
# Import core libraries for data handling and visualisation
import numpy as np              # for working with multi-dimensional arrays
import pandas as pd             # for data manipulation and analysis
import matplotlib.pyplot as plt # plotting library
from sklearn.feature_selection import mutual_info_classif
import seaborn as sns


## 2. Load the Dataset

In [ ]:
# Mount Google Drive so the dataset can be accessed from Colab
from google.colab import drive
drive.mount('/content/drive')

# Load the dataset from Google Drive
# Make sure the path matches exactly where the file is stored in your Drive
road_casualty_dataset = pd.read_csv('/content/drive/MyDrive/Machine Learning Project/dft-road-casualty-statistics-collision-2024.csv')


### Preview the Data

In [ ]:
# Preview the first few rows of the dataframe
road_casualty_dataset.head(3)


### Dataset Shape

In [ ]:
# Check the dataset dimensions (rows, columns)
road_casualty_dataset.shape


## 3. Data Cleaning
### Drop Irrelevant / Leaky Columns

In [ ]:
# Drop columns that are irrelevant or unsafe to use as predictors,
# as specified in the project brief
df = road_casualty_dataset
df.drop(columns=[
    'collision_index',                    # Unique alphanumeric identifier for the crash; the model could
                                           # use it to memorise specific crashes, artificially inflating
                                           # validation accuracy while failing to generalise
    'collision_year',                     # Dataset only covers 2024, so this column has zero variance
                                           # and provides no predictive value
    'collision_ref_no',                   # Unique alphanumeric identifier for the crash (see collision_index)
    'location_easting_osgr',
    'location_northing_osgr',
    'police_force',
    'local_authority_district',
    'local_authority_ons_district',
    'local_authority_highway',
    'local_authority_highway_current',
    'first_road_number',
    'junction_detail_historic',
    'pedestrian_crossing_human_control_historic',
    'pedestrian_crossing_physical_facilities_historic',
    'carriageway_hazards_historic',
    'second_road_number',
    'did_police_officer_attend_scene_of_accident', # Direct record of the post-collision emergency response
    'lsoa_of_accident_location',
    'number_of_casualties',
    'number_of_vehicles',                 # Exact counts of casualties/vehicles are typically confirmed only
                                           # after emergency services reach the scene, so using them would leak
                                           # post-collision information
    'enhanced_severity_collision',
    'collision_injury_based',             # Alternative medical/administrative classification of the crash
                                           # outcome; effectively a proxy for the target variable (collision_severity)
    'collision_adjusted_severity_serious',
    'collision_adjusted_severity_slight', # Statistically adjusted severity metrics calculated well after the
                                           # collision; including these would give the model the answer key
], inplace=True)


### Handle Missing Values & Compute Mutual Information

In [ ]:
# 1. Define the codes that represent missing/unknown data in this dataset
missing_codes = [-1, 9, 99]

# 2. Replace these specific numeric codes with true NaN values
df.replace(missing_codes, np.nan, inplace=True)

# 3. Impute the NaN values with the mode (most frequent value) for each column
# We use a loop to fill NaNs column by column
for column in df.columns:
    if df[column].isnull().any():
        mode_value = df[column].mode()[0]
        df[column].fillna(mode_value, inplace=True)

# 4. Now the data is ready. Isolate the target and features
# (ensure string columns like 'time' are dropped or encoded first)
# Assuming 'date' and 'time' have already been processed or temporarily dropped
X = df.select_dtypes(include=[np.number]).drop(columns=['collision_severity'])
y = df['collision_severity']

# 5. Calculate and display Mutual Information scores for every numeric feature
mi_scores = mutual_info_classif(X, y, random_state=42)

mi_results = pd.DataFrame({
    'Feature': X.columns,
    'MI_Score': mi_scores
}).sort_values(by='MI_Score', ascending=False).reset_index(drop=True)

print(mi_results)


### Drop Zero-MI Features

In [ ]:
# Drop features that yielded a Mutual Information score of 0.0
df.drop(columns=[
    'pedestrian_crossing',
    'carriageway_hazards',
    'special_conditions_at_site'
], inplace=True)


## 4. Visualise Mutual Information Scores

In [ ]:
# Ensure the results are sorted from highest to lowest MI score
mi_results = mi_results.sort_values(by='MI_Score', ascending=False)

# Set up the matplotlib figure size
plt.figure(figsize=(14, 8))

# 1. Create a bar plot to show the individual scores
sns.barplot(
    x='MI_Score',
    y='Feature',
    data=mi_results,
    palette='viridis',
    orient='h' # Horizontal bars are easier to read with long feature names
)

# 2. Add a line plot to help visually identify the "elbow" drop-off
# We plot the line on the same axes, using the index as the y-coordinate
plt.plot(
    mi_results['MI_Score'],
    range(len(mi_results)),
    color='red',
    marker='o',
    linestyle='-',
    linewidth=2,
    markersize=5,
    label='MI Trend (Elbow)'
)

# 3. Add labels, title, and formatting for the report
plt.title('Mutual Information Scores for Collision Severity', fontsize=16, fontweight='bold')
plt.xlabel('Mutual Information Score', fontsize=12)
plt.ylabel('Feature', fontsize=12)
plt.grid(axis='x', linestyle='--', alpha=0.7)
plt.legend()

# Adjust layout so feature names fit on the screen, then display the plot
plt.tight_layout()
plt.show()


## 5. Exploratory Data Analysis

In [ ]:
# Set the overall visual style for the plots
sns.set_theme(style="whitegrid")

# ==========================================
# 1. Grouped Count Plot (Raw Frequency)
# Good for seeing the total volume of collisions across categories
# ==========================================
plt.figure(figsize=(10, 6))
sns.countplot(
    data=df,
    x='urban_or_rural_area',
    hue='collision_severity',
    palette='viridis'
)
plt.title('Collision Severity Counts by Urban/Rural Area', fontsize=14, fontweight='bold')
plt.xlabel('Area Type (e.g., 1=Urban, 2=Rural)', fontsize=12)
plt.ylabel('Number of Collisions', fontsize=12)
plt.legend(title='Severity\n(0=Slight, 1=Serious, 2=Fatal)')
plt.tight_layout()
plt.show()

# ==========================================
# 2. 100% Stacked Bar Chart (Proportions)
# Crucial for imbalanced datasets to see if a category increases the *rate* of fatal/serious crashes
# ==========================================
# Calculate the percentage of each severity class within each trunk road category
crosstab_prop = pd.crosstab(
    df['trunk_road_flag'],
    df['collision_severity'],
    normalize='index' # Normalises across rows to create percentages
)

# Plot as a stacked bar chart
ax = crosstab_prop.plot(
    kind='bar',
    stacked=True,
    figsize=(10, 6),
    colormap='viridis'
)
plt.title('Proportion of Collision Severities by Trunk Road Flag', fontsize=14, fontweight='bold')
plt.xlabel('Trunk Road Flag (e.g., 1=Trunk, 2=Non-Trunk)', fontsize=12)
plt.ylabel('Proportion', fontsize=12)
plt.legend(title='Severity\n(0=Slight, 1=Serious, 2=Fatal)', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

# ==========================================
# 3. Violin Plot (Distribution of Numeric/Ordinal Features)
# Better than a box plot here because it shows the density of speed limits for each severity
# ==========================================
plt.figure(figsize=(10, 6))
sns.violinplot(
    data=df,
    x='collision_severity',
    y='speed_limit',
    palette='viridis',
    inner='quartile' # Shows the median and quartiles inside the violin
)
plt.title('Distribution of Speed Limits Across Collision Severities', fontsize=14, fontweight='bold')
plt.xlabel('Collision Severity (0=Slight, 1=Serious, 2=Fatal)', fontsize=12)
plt.ylabel('Speed Limit (mph)', fontsize=12)
plt.tight_layout()
plt.show()


## 6. Feature Engineering
### Weekend Flag (`day_of_week` → `is_weekend`)

In [ ]:
# 1. Create the new engineered feature
# Note: in the UK Road Safety dataset, 'day_of_week' is typically encoded as 1 (Sunday) through 7 (Saturday).
# We map Sunday (1) and Saturday (7) to 1 (Weekend), and all other days to 0 (Weekday).
df['is_weekend'] = df['day_of_week'].apply(lambda x: 1 if x in [1, 7] else 0)

# 2. Isolate the target variable
y = df['collision_severity']

# 3. Isolate the features into the 2D format required by scikit-learn
X_original = df[['day_of_week']]
X_engineered = df[['is_weekend']]

# 4. Calculate Mutual Information scores for both
# random_state=42 ensures the results are exactly reproducible for the report
mi_original = mutual_info_classif(X_original, y, random_state=42)[0]
mi_engineered = mutual_info_classif(X_engineered, y, random_state=42)[0]

# 5. Display the comparison
print("--- Feature Engineering MI Comparison ---")
print(f"Original Feature ('day_of_week') MI Score:   {mi_original:.6f}")
print(f"Engineered Feature ('is_weekend') MI Score:  {mi_engineered:.6f}")

# 6. Automated justification
if mi_engineered >= mi_original:
    print("\nResult: SUCCESS")
    print("The engineered feature successfully distilled the signal. You have mathematical justification to drop 'day_of_week' and keep 'is_weekend'.")
else:
    print("\nResult: INFORMATION LOST")
    print("The engineered feature lost predictive power compared to the original. You should either keep 'day_of_week' or try a different engineering approach.")

# 7. Permanently drop the original 'day_of_week' column
# This prevents multicollinearity and redundancy in the models
df.drop(columns=['day_of_week'], inplace=True)

# Verify the changes
print("Successfully engineered 'is_weekend'.")
print("Remaining columns:", df.columns.tolist())


### Adverse Weather Flag (`weather_conditions` → `is_adverse_weather`)

In [ ]:
# 1. Create the new engineered feature
# Map 1 (fine, no high winds) to 0 (normal). Map everything else to 1 (adverse).
df['is_adverse_weather'] = df['weather_conditions'].apply(lambda x: 0 if x == 1 else 1)

# 2. Isolate the target variable
y = df['collision_severity']

# 3. Isolate the features for comparison
X_original_weather = df[['weather_conditions']]
X_engineered_weather = df[['is_adverse_weather']]

# 4. Calculate Mutual Information scores
mi_original_weather = mutual_info_classif(X_original_weather, y, random_state=42)[0]
mi_engineered_weather = mutual_info_classif(X_engineered_weather, y, random_state=42)[0]

# 5. Display the comparison
print("--- Weather Feature Engineering MI Comparison ---")
print(f"Original Feature ('weather_conditions') MI Score:   {mi_original_weather:.6f}")
print(f"Engineered Feature ('is_adverse_weather') MI Score: {mi_engineered_weather:.6f}")

# 6. Automated justification
if mi_engineered_weather >= mi_original_weather:
    print("\nResult: SUCCESS")
    print("The binary flag successfully concentrated the signal. You should drop 'weather_conditions' and keep 'is_adverse_weather'.")
else:
    print("\nResult: INFORMATION LOST")
    print("The granular categories contained valuable nuance. Keep 'weather_conditions' and drop the engineered flag.")

# Drop the engineered feature since it underperformed
df.drop(columns=['is_adverse_weather'], inplace=True)


### Controlled Junction Flag (`junction_control` → `controlled_junction`)

In [ ]:
# 1. Create the new engineered feature
# Map 1, 2 and 3 (police, lights, stop signs) to 1 (controlled).
# Map everything else (give way, uncontrolled, or not at junction) to 0.
df['controlled_junction'] = df['junction_control'].apply(lambda x: 1 if x in [1, 2, 3] else 0)

# 2. Isolate the target variable
y = df['collision_severity']

# 3. Isolate the features for comparison
X_original_junc = df[['junction_control']]
X_engineered_junc = df[['controlled_junction']]

# 4. Calculate Mutual Information scores
# random_state=42 is used for reproducibility in the report
mi_original_junc = mutual_info_classif(X_original_junc, y, random_state=42)[0]
mi_engineered_junc = mutual_info_classif(X_engineered_junc, y, random_state=42)[0]

# 5. Display the comparison
print("--- Junction Feature Engineering MI Comparison ---")
print(f"Original Feature ('junction_control') MI Score:    {mi_original_junc:.6f}")
print(f"Engineered Feature ('controlled_junction') MI Score: {mi_engineered_junc:.6f}")

# 6. Automated justification
if mi_engineered_junc >= mi_original_junc:
    print("\nResult: SUCCESS")
    print("The binary flag successfully concentrated the signal. You should drop 'junction_control' and keep 'controlled_junction'.")
else:
    print("\nResult: INFORMATION LOST")
    print("The granular categories contained valuable nuance. Keep 'junction_control' and drop the engineered flag.")

df.drop(columns=['junction_control'], inplace=True)


### Darkness Flag (`light_conditions` → `is_darkness`)

In [ ]:
# Create the engineered feature: map 1 (daylight) to 0, and all darkness variants to 1
df['is_darkness'] = df['light_conditions'].apply(lambda x: 0 if x == 1 else 1)

# Isolate target and features
y = df['collision_severity']
X_orig_light = df[['light_conditions']]
X_eng_light = df[['is_darkness']]

# Calculate Mutual Information scores
mi_orig_light = mutual_info_classif(X_orig_light, y, random_state=42)[0]
mi_eng_light = mutual_info_classif(X_eng_light, y, random_state=42)[0]

print("--- Light Conditions MI Comparison ---")
print(f"Original ('light_conditions'): {mi_orig_light:.6f}")
print(f"Engineered ('is_darkness'):    {mi_eng_light:.6f}")

if mi_eng_light >= mi_orig_light:
    print("Result: SUCCESS - Keep 'is_darkness', drop 'light_conditions'.\n")
else:
    print("Result: INFO LOST - Keep 'light_conditions', drop 'is_darkness'.\n")

df.drop(columns=['is_darkness'], inplace=True)


### Adverse Surface Flag (`road_surface_conditions` → `is_adverse_surface`)

In [ ]:
# Create the engineered feature: map 1 (dry) to 0, everything else to 1
df['is_adverse_surface'] = df['road_surface_conditions'].apply(lambda x: 0 if x == 1 else 1)

# Isolate features
X_orig_surf = df[['road_surface_conditions']]
X_eng_surf = df[['is_adverse_surface']]

# Calculate Mutual Information scores
mi_orig_surf = mutual_info_classif(X_orig_surf, y, random_state=42)[0]
mi_eng_surf = mutual_info_classif(X_eng_surf, y, random_state=42)[0]

print("--- Road Surface MI Comparison ---")
print(f"Original ('road_surface_conditions'): {mi_orig_surf:.6f}")
print(f"Engineered ('is_adverse_surface'):    {mi_eng_surf:.6f}")

if mi_eng_surf >= mi_orig_surf:
    print("Result: SUCCESS - Keep 'is_adverse_surface', drop 'road_surface_conditions'.\n")
else:
    print("Result: INFO LOST - Keep 'road_surface_conditions', drop 'is_adverse_surface'.\n")

df.drop(columns=['is_adverse_surface'], inplace=True)


### High Speed Flag (`speed_limit` → `is_high_speed`)

In [ ]:
# Create the engineered feature: map speed >= 50mph to 1, < 50mph to 0
df['is_high_speed'] = df['speed_limit'].apply(lambda x: 1 if pd.notnull(x) and x >= 50 else 0)

# Isolate features
X_orig_speed = df[['speed_limit']]
X_eng_speed = df[['is_high_speed']]

# Calculate Mutual Information scores
mi_orig_speed = mutual_info_classif(X_orig_speed, y, random_state=42)[0]
mi_eng_speed = mutual_info_classif(X_eng_speed, y, random_state=42)[0]

print("--- Speed Limit MI Comparison ---")
print(f"Original ('speed_limit'): {mi_orig_speed:.6f}")
print(f"Engineered ('is_high_speed'):    {mi_eng_speed:.6f}")

if mi_eng_speed >= mi_orig_speed:
    print("Result: SUCCESS - Keep 'is_high_speed', drop 'speed_limit'.\n")
else:
    print("Result: INFO LOST - Keep 'speed_limit', drop 'is_high_speed'.\n")

df.drop(columns=['is_high_speed'], inplace=True)


### Combined "Counts" Feature (Total Adverse Conditions)

In [ ]:
# 1. Create temporary binary flags for adverse conditions
# In the UK DfT dataset, '1' generally represents the safe/normal condition
# (1 = fine weather, 1 = daylight, 1 = dry surface)
bad_weather = df['weather_conditions'].apply(lambda x: 0 if x == 1 else 1)
bad_light = df['light_conditions'].apply(lambda x: 0 if x == 1 else 1)
bad_surface = df['road_surface_conditions'].apply(lambda x: 0 if x == 1 else 1)

# 2. Apply the "Counts" technique by summing the flags
# This creates a new feature ranging from 0 (perfect conditions) to 3 (terrible conditions)
df['total_adverse_conditions'] = bad_weather + bad_light + bad_surface

# 3. Isolate the target and the features
y = df['collision_severity']
X_counts = df[['total_adverse_conditions']]

# 4. Calculate the Mutual Information score for the new count feature
mi_counts = mutual_info_classif(X_counts, y, random_state=42)[0]

# 5. Display the result
print("--- 'Counts' Feature Engineering MI Test ---")
print(f"Engineered Feature ('total_adverse_conditions') MI Score: {mi_counts:.6f}")

# Compare against the original max MI score from these three columns
# (weather was ~0.0052, surface was ~0.0053, light was ~0.0076)
if mi_counts > 0.007673:
    print("\nResult: SUCCESS")
    print("The combined count feature outperforms all individual environmental features. Keep 'total_adverse_conditions'.")
else:
    print("\nResult: MODERATE / FAILED")
    print("The combined count did not outperform the best individual feature ('light_conditions').")
    print("You should drop 'total_adverse_conditions' and rely on the original granular columns.")
    # df.drop(columns=['total_adverse_conditions'], inplace=True) # Uncomment to drop


### Group Transform: Frequency Encoding (`first_road_class`)

In [ ]:
# 1. Define the target feature for the group transform
feature_to_transform = 'first_road_class'
original_mi_score = 0.007573  # from the previous MI list

# 2. Apply the Group Transform (Frequency Encoding)
# Groups by the road class, counts the occurrences, and divides by the total row
# count to get the proportion of accidents for each road class
df['first_road_class_freq'] = (
    df.groupby(feature_to_transform)[feature_to_transform]
    .transform('count') / df[feature_to_transform].count()
)

# 3. Isolate the target and the engineered feature
y = df['collision_severity']
X_freq = df[['first_road_class_freq']]

# 4. Calculate the Mutual Information score for the new frequency feature
mi_freq = mutual_info_classif(X_freq, y, random_state=42)[0]

# 5. Display the comparison
print("--- 'Group Transforms' (Frequency Encoding) MI Test ---")
print(f"Original Feature ('{feature_to_transform}') MI Score:   {original_mi_score:.6f}")
print(f"Engineered Feature ('first_road_class_freq') MI Score: {mi_freq:.6f}")

# 6. Automated justification
if mi_freq > original_mi_score:
    print("\nResult: SUCCESS")
    print(f"Replacing the raw categories with their frequency volume improved the predictive signal. You should drop '{feature_to_transform}' and keep the frequency version.")
else:
    print("\nResult: INFORMATION LOST")
    print(f"The specific road classes contain nuance that is lost when reduced to just their frequencies. Drop 'first_road_class_freq' and keep the original.")
    # df.drop(columns=['first_road_class_freq'], inplace=True) # Uncomment to drop


### Combination Feature: Road Type + Speed Limit

In [ ]:
# 1. Combine two interacting features into a single string
# Example: road type 6 and speed limit 30 becomes "6_30.0"
df['road_speed_combo'] = df['road_type'].astype(str) + "_" + df['speed_limit'].astype(str)

# 2. Convert the new text string into numeric codes so scikit-learn can process it
df['road_speed_combo_encoded'] = df['road_speed_combo'].astype('category').cat.codes

# 3. Isolate target and the new encoded feature
y = df['collision_severity']
X_combo = df[['road_speed_combo_encoded']]

# 4. Calculate Mutual Information
mi_combo = mutual_info_classif(X_combo, y, random_state=42)[0]

# Retrieve the highest original MI score from the two source features
# (road_type was ~0.010678, speed_limit was ~0.009557)
max_original_mi = max(0.010678, 0.009557)

# 5. Display the comparison
print("--- 'Combinations' Feature Engineering MI Test ---")
print(f"Original 'road_type' MI Score:   0.010678")
print(f"Original 'speed_limit' MI Score: 0.009557")
print(f"Engineered Combo MI Score:       {mi_combo:.6f}")

# 6. Automated justification
if mi_combo > max_original_mi:
    print("\nResult: SUCCESS")
    print("The combination captured a powerful interaction effect. Keep the new combo feature!")
    print("Note: To avoid multicollinearity, you would drop 'road_type' and 'speed_limit'.")
else:
    print("\nResult: NO IMPROVEMENT")
    print("The combination did not add predictive value beyond the original features.")
    print("Drop the combo feature and keep the originals.")
    # df.drop(columns=['road_speed_combo', 'road_speed_combo_encoded'], inplace=True) # Uncomment to drop


### Combination Feature: Urban/Rural Area + Trunk Road Flag

In [ ]:
# 1. Combine the top two features into a single string
df['urban_trunk_combo'] = df['urban_or_rural_area'].astype(str) + "_" + df['trunk_road_flag'].astype(str)

# 2. Convert to numeric codes for the MI calculation
df['urban_trunk_combo_encoded'] = df['urban_trunk_combo'].astype('category').cat.codes

# 3. Isolate target and the new encoded feature
y = df['collision_severity']
X_urban_trunk = df[['urban_trunk_combo_encoded']]

# 4. Calculate Mutual Information
mi_urban_trunk = mutual_info_classif(X_urban_trunk, y, random_state=42)[0]

# Retrieve the highest original MI score from the two source features
max_orig_mi = max(0.017232, 0.015024)

# 5. Display the comparison
print("--- 'Urban/Trunk' Combination MI Test ---")
print(f"Original 'trunk_road_flag' MI:     0.017232")
print(f"Original 'urban_or_rural_area' MI: 0.015024")
print(f"Engineered Combo MI Score:         {mi_urban_trunk:.6f}")

# 6. Automated justification
if mi_urban_trunk > max_orig_mi:
    print("\nResult: SUCCESS")
    print("The macro-environment combination outperformed the individual features. Keep 'urban_trunk_combo'!")
else:
    print("\nResult: NO IMPROVEMENT")
    print("The combination diluted the signal. Keep 'urban_or_rural_area' and 'trunk_road_flag' separate.")
    # df.drop(columns=['urban_trunk_combo', 'urban_trunk_combo_encoded'], inplace=True)


### Combination Feature: Weather + Road Surface (Environmental Hazard)

In [ ]:
# 1. Combine the two environmental features into a single string
# Example: weather 1 (fine) and surface 4 (ice) becomes "1_4"
df['env_hazard_combo'] = df['weather_conditions'].astype(str) + "_" + df['road_surface_conditions'].astype(str)

# 2. Convert to numeric category codes for scikit-learn
df['env_hazard_combo_encoded'] = df['env_hazard_combo'].astype('category').cat.codes

# 3. Isolate target and the new encoded feature
y = df['collision_severity']
X_env_hazard = df[['env_hazard_combo_encoded']]

# 4. Calculate Mutual Information
mi_env_hazard = mutual_info_classif(X_env_hazard, y, random_state=42)[0]

# Retrieve the highest original MI score from the two source features
# (road_surface_conditions was ~0.005300, weather_conditions was ~0.005200)
max_env_orig_mi = max(0.005300, 0.005200)

# 5. Display the comparison
print("--- 'Environmental Hazard' Combination MI Test ---")
print(f"Original 'road_surface' MI: {0.005300:.6f}")
print(f"Original 'weather' MI:      {0.005200:.6f}")
print(f"Engineered Combo MI Score:  {mi_env_hazard:.6f}")

# 6. Automated justification
if mi_env_hazard > max_env_orig_mi:
    print("\nResult: SUCCESS")
    print("The specific combinations of weather and surface conditions are highly predictive.")
    print("Keep 'env_hazard_combo'. Remember to drop the original weather and surface columns to prevent redundancy.")
else:
    print("\nResult: NO IMPROVEMENT")
    print("The combination created too many granular categories without adding strong signal.")
    print("Drop the combo feature and retain the original 'weather_conditions' and 'road_surface_conditions' columns.")
    # df.drop(columns=['env_hazard_combo', 'env_hazard_combo_encoded'], inplace=True) # Uncomment to revert


### Combination Feature: First + Second Road Class (Junction Hierarchy)

In [ ]:
# 1. Combine the two road class features into a single string
# Example: first class 1 (motorway) and second class 6 (unclassified) becomes "1_6"
df['junction_hierarchy_combo'] = df['first_road_class'].astype(str) + "_" + df['second_road_class'].astype(str)

# 2. Convert to numeric category codes for scikit-learn
df['junction_hierarchy_combo_encoded'] = df['junction_hierarchy_combo'].astype('category').cat.codes

# 3. Isolate target and the new encoded feature
y = df['collision_severity']
X_junc_hierarchy = df[['junction_hierarchy_combo_encoded']]

# 4. Calculate Mutual Information
mi_junc_hierarchy = mutual_info_classif(X_junc_hierarchy, y, random_state=42)[0]

# Retrieve the highest original MI score from the two source features
# (second_road_class was ~0.010121, first_road_class was ~0.007573)
max_junc_orig_mi = max(0.010121, 0.007573)

# 5. Display the comparison
print("--- 'Junction Hierarchy' Combination MI Test ---")
print(f"Original 'second_road_class' MI: {0.010121:.6f}")
print(f"Original 'first_road_class' MI:  {0.007573:.6f}")
print(f"Engineered Combo MI Score:       {mi_junc_hierarchy:.6f}")

# 6. Automated justification
if mi_junc_hierarchy > max_junc_orig_mi:
    print("\nResult: SUCCESS")
    print("The specific intersection pairings provide a stronger predictive signal.")
    print("Keep 'junction_hierarchy_combo' and drop the original individual class columns.")
else:
    print("\nResult: NO IMPROVEMENT")
    print("The combination created high cardinality (too many unique pairs) without adding value.")
    print("Drop the combo feature and keep 'first_road_class' and 'second_road_class' separate.")
    # df.drop(columns=['junction_hierarchy_combo', 'junction_hierarchy_combo_encoded'], inplace=True) # Uncomment to revert


## 7. Finalise Dataset & Export

In [ ]:
# 1. Feature engineer the date and time
# Machine learning models cannot process raw "YYYY-MM-DD" or "HH:MM" strings.
# We extract 'hour_of_day' and 'month' as they often hold strong predictive signals
# for traffic collisions (e.g., rush hour traffic, winter weather).
df['hour_of_day'] = pd.to_datetime(df['time'], format='%H:%M', errors='coerce').dt.hour
df['month'] = pd.to_datetime(df['date'], format='%d/%m/%Y', errors='coerce').dt.month

# Fill any rare parsing errors (NaNs) with the median
df['hour_of_day'] = df['hour_of_day'].fillna(df['hour_of_day'].median())
df['month'] = df['month'].fillna(df['month'].median())

# 2. Define the columns to drop
# This includes the raw date/time text and all the temporary MI test combinations
columns_to_drop = [
    'date',
    'time',
    'total_adverse_conditions',
    'first_road_class_freq',
    'road_speed_combo',
    'road_speed_combo_encoded',
    'urban_trunk_combo',
    'urban_trunk_combo_encoded',
    'env_hazard_combo',
    'env_hazard_combo_encoded',
    'junction_hierarchy_combo',
    'junction_hierarchy_combo_encoded'
]

# 3. Drop them from the DataFrame safely
# Use a list comprehension to only drop columns that actually exist, preventing errors
df.drop(columns=[col for col in columns_to_drop if col in df.columns], inplace=True)

# 4. Re-export the cleaned dataset to Google Drive
df.to_csv('/content/drive/MyDrive/Machine Learning Project/cleaned_collision_data.csv', index=False)

print("T1.1 Cleanup Complete. Clean data successfully re-exported to Google Drive!")
print("Final curated columns:", df.columns.tolist())
